<a href="https://colab.research.google.com/github/mrclassic7499/DSA0302-NLP/blob/main/ASSIGNMENTS/NLP_ASSIGNMENT_TOOL_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DSA03-NATURAL LANGUAGE PROCESSING: ASSIGNMENT-4
## Meaning-Based Enquiry & Document Search Layer for a Government Pension Helpdesk

### Task 1 & 2: Semantic Pipeline Setup & Verification
We define basic rule-matching schemas and set up the foundation of our pipeline, representing logical meaning structures.

### Task 1: Pipeline Architecture Diagram

```
+-----------------------+
|  Natural Language     |
|  Enquiry / Query      |
+-----------+-----------+
            |
            v
+-----------+-----------+        +--------------------------+
|  Syntax-Driven        | <----> | Custom Feature Grammar   |
|  Semantic Parser      |        | (Compositional Lambda)   |
+-----------+-----------+        +--------------------------+
            |
            v
+-----------+-----------+        +--------------------------+
| Word Sense            | <----> | Domain Sense Inventory   |
| Disambiguator (WSD)   |        | (Lesk / ML Classifiers)  |
+-----------+-----------+        +--------------------------+
            |
            v
+-----------+-----------+        +--------------------------+
| First-Order Logic     | <----> | Knowledge Base           |
| Inference Engine      |        | (FOPC Rules / Model)     |
+-----------+-----------+        +--------------------------+
            |
            v
+-----------+-----------+        +--------------------------+
| Information           | <----> | Inverted Index /         |
| Retrieval (IR) Layer  |        | TF-IDF over Circulars    |
+-----------+-----------+        +--------------------------+
            |
            v
+-----------+-----------+
| Final Answer +        |
| Matched Circular Docs |
+-----------------------+
```

In [12]:
import nltk
from nltk.sem import Expression
from nltk.inference import ResolutionProver
import numpy as np
import pandas as pd

# Quick initialization of standard tools
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

print("Setup Completed: Ready to translate and verify logic representations.")

Setup Completed: Ready to translate and verify logic representations.


### Task 3: First-Order Predicate Calculus (FOPC) & Model Evaluation

#### Task 3: In-Depth Model Evaluation
Below we build a concrete evaluation model using domains, predicates, and constant individuals to verify logical formulas mathematically.

In [13]:
read_expr = Expression.fromstring

# 1. Define standard rules (FOPC knowledge base)
kb = [
    read_expr('all x (CompletedTenYears(x) -> QualifiesPension(x))'),
    read_expr('all x (IsDependent(x) & RetiredEmployee(y) & FamilyPensionEligible(y) -> FamilyPension(x))'),
    read_expr('all x (BreakInService(x) & ApprovedExemption(x) -> CountsAsQualifying(x))')
]

# 2. Add specific assertions/facts
facts = [
    read_expr('CompletedTenYears(john)'),
    read_expr('BreakInService(break1)'),
    read_expr('ApprovedExemption(break1)')
]

# Combined Knowledge Base
full_kb = kb + facts

# Prover evaluation
goal1 = read_expr('QualifiesPension(john)')
result1 = ResolutionProver().prove(goal1, full_kb, verbose=False)
print(f"Does John qualify for pension? {result1}")

goal2 = read_expr('CountsAsQualifying(break1)')
result2 = ResolutionProver().prove(goal2, full_kb, verbose=False)
print(f"Does break1 count as qualifying service? {result2}")

Does John qualify for pension? True
Does break1 count as qualifying service? True


In [14]:
import nltk
from nltk.sem import Valuation, Model

# Define domain of discourse and valuation mapping
v = [('john', 'b1'), ('break1', 'b2'), ('CompletedTenYears', {'b1'}), ('BreakInService', {'b2'}), ('ApprovedExemption', {'b2'})]
val = Valuation(v)
dom = val.domain
m = Model(dom, val)
g = nltk.sem.Assignment(dom)

# Evaluate some logical formulas in our defined model
formula1 = 'CompletedTenYears(john)'
formula2 = 'BreakInService(break1)'
formula3 = 'exists x (BreakInService(x) & ApprovedExemption(x))'

print("Evaluating FOPC Formula 1:", m.evaluate(formula1, g))
print("Evaluating FOPC Formula 2:", m.evaluate(formula2, g))
print("Evaluating FOPC Formula 3 (Existential):", m.evaluate(formula3, g))

Evaluating FOPC Formula 1: True
Evaluating FOPC Formula 2: True
Evaluating FOPC Formula 3 (Existential): True


### Task 4 & 5: Syntax-Driven Semantic Analysis (Feature Grammar with Semantic Attachments)

### Task 4: Representing Linguistically Relevant Concepts
We structure relations for domain-specific categories (e.g., employee, pensioner, dependent) and use Neo-Davidsonian event variables to express actions.

In [15]:
# Representing event variables: e.g., 'John retired in 2019'
# Event e: Retire(e) & Agent(e, John) & Time(e, 2019)
event_representation = """
exists e (RetireEvent(e) & Agent(e, john) & EventTime(e, y2019))
"""
print("Neo-Davidsonian Event Representation for 'John retired in 2019':")
print(event_representation.strip())

Neo-Davidsonian Event Representation for 'John retired in 2019':
exists e (RetireEvent(e) & Agent(e, john) & EventTime(e, y2019))


In [16]:
from nltk.parse import FeatureEarleyChartParser

# We define a toy semantic grammar in NLTK format
grammar_str = """
% start S
S[SEM=<?vp(?subj)>] -> NP[SEM=?subj] VP[SEM=?vp]
NP[SEM=?nom] -> PropN[SEM=?nom]
VP[SEM=?v] -> IV[SEM=?v]

PropN[SEM=<john>] -> 'John'
IV[SEM=<\\x.QualifiesPension(x)>] -> 'qualifies'
"""

grammar = nltk.grammar.FeatureGrammar.fromstring(grammar_str)
parser = FeatureEarleyChartParser(grammar)

sentence = 'John qualifies'.split()
for tree in parser.parse(sentence):
    print("Parse Tree:")
    print(tree)
    print("Resulting Logical Form:")
    print(tree.label()['SEM'].simplify())

Parse Tree:
(S[SEM=<QualifiesPension(john)>]
  (NP[SEM=<john>] (PropN[SEM=<john>] John))
  (VP[SEM=<\x.QualifiesPension(x)>]
    (IV[SEM=<\x.QualifiesPension(x)>] qualifies)))
Resulting Logical Form:
QualifiesPension(john)


### Task 6 & 7: Word Sense Disambiguation (WSD) via Lesk & Supervised Classifier

### Task 6: Lexemes and Their Senses
We document custom domain term senses such as `service` (employment duration vs helpdesk function), `award` (granting decision vs judicial determination), and `dependent` (family relative vs relational status).

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# 1. Simplified Lesk Algorithm for disambiguating 'service'
# Senses: sense_1 = 'military or civil employment period', sense_2 = 'customer/pension helpdesk assistance'
sense_inventory = {
    'service_employment': set(['years', 'work', 'retirement', 'duration', 'completed', 'tenure']),
    'service_helpdesk': set(['enquiry', 'pensioner', 'reply', 'question', 'assistance'])
}

def lesk_disambiguate(context_words):
    best_sense = None
    max_overlap = -1
    for sense, signature in sense_inventory.items():
        overlap = len(set(context_words).intersection(signature))
        if overlap > max_overlap:
            max_overlap = overlap
            best_sense = sense
    return best_sense, max_overlap

context_1 = ['completed', 'ten', 'years', 'of', 'service', 'duration']
print("Lesk output for context 1:", lesk_disambiguate(context_1))

# 2. Supervised WSD Classifier setup
train_contexts = [
    ("he completed ten years of service", "service_employment"),
    ("she retired after long service", "service_employment"),
    ("the customer service answered quickly", "service_helpdesk"),
    ("helpdesk service was excellent", "service_helpdesk")
]

X_train, y_train = zip(*[(text, label) for text, label in train_contexts])
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(X_train)
clf = MultinomialNB().fit(X_vec, y_train)

# Test WSD model
test_text = ["pensioner service was helpful"]
test_vec = vectorizer.transform(test_text)
print(f"Supervised WSD prediction for '{test_text[0]}':", clf.predict(test_vec)[0])

Lesk output for context 1: ('service_employment', 3)
Supervised WSD prediction for 'pensioner service was helpful': service_helpdesk


### Task 8 & 9: Information Retrieval and Inverted Index over Circulars

### Task 8 & 9: Inverted Index & Query Expansion
We expand a plain retrieval query with sense-disambiguated tokens to filter irrelevant documents and measure downstream precision benefits.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sample Synthetic Circular Documents
circulars = [
    "Circular 101: Employees completing ten years of service are eligible for pension benefits.",
    "Circular 102: Family pension rules and dependency criteria for surviving dependents.",
    "Circular 103: Service break and rules governing approved exemptions or service interruption."
]

# Initialize TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(circulars)

# Query search function
def search_circulars(query):
    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    results = sorted(list(enumerate(similarity)), key=lambda x: x[1], reverse=True)
    return results

query = "qualifying service break rules"
print(f"Search results for: '{query}':")
for idx, score in search_circulars(query):
    print(f"Doc {idx+1} (Score: {score:.4f}): {circulars[idx]}")

Search results for: 'qualifying service break rules':
Doc 3 (Score: 0.6127): Circular 103: Service break and rules governing approved exemptions or service interruption.
Doc 1 (Score: 0.1438): Circular 101: Employees completing ten years of service are eligible for pension benefits.
Doc 2 (Score: 0.1438): Circular 102: Family pension rules and dependency criteria for surviving dependents.


In [19]:
# Query expansion based on WSD outcomes
def expand_query_with_sense(query, sense_tag):
    return f"{query} {sense_tag}"

expanded_q = expand_query_with_sense("pension qualification service", "service_employment")
print("Original Query:", "pension qualification service")
print("Sense-Aware Expanded Query:", expanded_q)

Original Query: pension qualification service
Sense-Aware Expanded Query: pension qualification service service_employment


### Task 10, 11 & 12: Day-4 Design Challenge - Adaptation to Aggregation, Dates & Rules

In [20]:
# We implement a temporal and numerical simulation layer to address the Day-4 Design Challenge.
# This helper function handles counting and date-logic boundaries over service tables.

def evaluate_eligibility_adapted(retirement_year, break_records, total_service_years):
    # 1. Rule versioning based on retirement date
    scheme_version = "Legacy" if retirement_year < 2020 else "Revised"

    # 2. Count and Aggregation
    # In the Revised scheme, approved breaks do not invalidate total qualifying service
    # but must be subtracted if they are unapproved.
    unapproved_breaks = sum(1 for b in break_records if not b.get('approved', False))
    adjusted_service_years = total_service_years - (unapproved_breaks * 0.5) # Penalty

    eligible = False
    if scheme_version == "Legacy":
        eligible = adjusted_service_years >= 10
    else:
        # Revised Scheme: threshold is lowered to 8 years if the candidate has no unapproved breaks
        threshold = 8 if unapproved_breaks == 0 else 10
        eligible = adjusted_service_years >= threshold

    return {
        "scheme_version": scheme_version,
        "adjusted_service_years": adjusted_service_years,
        "eligible_for_pension": eligible
    }

# Trace Case: Retired in 2021, 9 years of service, 1 unapproved break
result_case_1 = evaluate_eligibility_adapted(retirement_year=2021, break_records=[{'approved': False}], total_service_years=9)
print("Case 1 (2021 retirement, 1 unapproved break):", result_case_1)

# Trace Case: Retired in 2021, 9 years of service, 0 breaks
result_case_2 = evaluate_eligibility_adapted(retirement_year=2021, break_records=[], total_service_years=9)
print("Case 2 (2021 retirement, 0 breaks):", result_case_2)

Case 1 (2021 retirement, 1 unapproved break): {'scheme_version': 'Revised', 'adjusted_service_years': 8.5, 'eligible_for_pension': False}
Case 2 (2021 retirement, 0 breaks): {'scheme_version': 'Revised', 'adjusted_service_years': 9.0, 'eligible_for_pension': True}
